# **Telecom RAG Pipeline**


In [ ]:
!pip install pyngrok
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list
!sudo apt-get update -y
!sudo apt-get install ngrok -y

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic

In [ ]:
import nest_asyncio
import uvicorn
import threading
import traceback
import sqlite3
import shutil
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok, conf
from google.colab import userdata

# Applying nest_asyncio so Uvicorn plays nicely with Colab's event loop
nest_asyncio.apply()

SQLITE_PATH = "app.db"
def process_query(query):
    return "answer", [{"sop_id": "1", "vendor": "V", "severity": "S", "title": "T"}]

conn = sqlite3.connect(SQLITE_PATH, check_same_thread=False)
cursor = conn.cursor()

# 1. Define the API Schema
class QueryRequest(BaseModel):
    query: str

class SopResponse(BaseModel):
    sop_id: str
    vendor: str
    severity: str
    title: str

class QueryResponse(BaseModel):
    answer: str
    retrieved_sops: list[SopResponse]

# 2. Initialize FastAPI
app = FastAPI(title="NetRestore RAG API", version="1.0")

# 3. Define the POST Endpoint
@app.post("/ask", response_model=QueryResponse)
async def ask_netrestore(request: QueryRequest):
    try:
        print(f"\nReceived query from frontend: {request.query}")

        # Call pipeline
        ans, srcs = process_query(request.query)

        print("Query processed successfully! Sending back to frontend...")

        # Format the sources cleanly for the JSON response
        formatted_srcs = []
        for s in srcs:
            formatted_srcs.append(SopResponse(
                sop_id=s.get("sop_id", "N/A"),
                vendor=s.get("vendor", "Unknown"),
                severity=s.get("severity", "UNKNOWN"),
                title=s.get("title", "Untitled")
            ))

        return QueryResponse(answer=ans, retrieved_sops=formatted_srcs)

    except Exception as e:
        print("\nERROR IN PROCESS_QUERY:")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


# 4. Expose server to the internet

ngrok_exec = shutil.which("ngrok")
if not ngrok_exec:
    raise RuntimeError("Ngrok binary not found!")

pyngrok_config = conf.PyngrokConfig(ngrok_path=ngrok_exec)
conf.set_default(pyngrok_config)

ngrok_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_token, pyngrok_config=pyngrok_config)

# Close any existing tunnels to prevent errors
ngrok.kill()

public_url = ngrok.connect(8501, pyngrok_config=pyngrok_config).public_url

print(f"NETRESTORE API IS LIVE AT: {public_url}/ask")

# 5. Start the Server in a Background Thread
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8501)

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print("Background thread started. API is ready to receive requests!")

In [ ]:
# TELECOM NOC PROCEDURAL RAG
!pip install -q datasets sentence-transformers chromadb pandas rank_bm25 transformers accelerate bitsandbytes

import torch
import chromadb
import pandas as pd
import string
import textwrap
import gc
import numpy as np
import os
import shutil
import json
import sqlite3
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Accelerator: {device}")

In [ ]:
# DB_PATH = "./telecom_vector_db"
# SQLITE_PATH = "telecom_sops.db"

import uuid

DB_PATH = f"/tmp/telecom_vector_db_{uuid.uuid4().hex}"
SQLITE_PATH = f"/tmp/telecom_sops_{uuid.uuid4().hex}.db"

# 1. Clean up old runs
if os.path.exists(DB_PATH): shutil.rmtree(DB_PATH)
if os.path.exists(SQLITE_PATH): os.remove(SQLITE_PATH)

# 2. Initialize SQLite (for Document Store)
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sops (
        sop_id TEXT PRIMARY KEY,
        vendor TEXT,
        severity TEXT,
        full_json TEXT
    )
''')
conn.commit()
print("SQLite initialized.")

# 3. Initialize ChromaDB (for Vector Store)
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection(
    name="telecom_sops",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB initialized.")

# 4. Load Embedding Model
print("🔹 Loading SOTA Embedding Model: BAAI/bge-base-en-v1.5")
embedding_func = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

In [ ]:
print("Loading structured SOPs and rewriting for dense embeddings:")

with open("telecom_sops_full.json", "r") as f:
    dataset = json.load(f)

bm25_tokenized_corpus = []
batch_docs = []
batch_metadatas = []
batch_ids = []

seen_ids = set()

def simple_tokenize(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation)).split()

for sop in tqdm(dataset, desc="Indexing"):
    sop_id = sop["sop_id"]

    if sop_id in seen_ids:
        continue
    seen_ids.add(sop_id)

    vendor = sop.get("vendor", "Unknown")
    severity = sop.get("severity", "UNKNOWN")

    # Conversational Data Representation
    term = sop.get('title', '').replace('SOP for ', '').replace(' Disruption', '')
    prose_content = f"This is a {severity} severity Standard Operating Procedure (SOP) for {vendor} equipment. "
    prose_content += f"It resolves disruptions caused by {term}. The procedure involves the following steps: "

    steps_text = " ".join([f"Step {s.get('step_number')}: {s.get('action')}. Execute command: `{s.get('command')}`." for s in sop.get('steps', [])])
    prose_content += steps_text

    # Override the original rigid search_content
    sop["search_content"] = prose_content

    # 1. Insert into SQLite (Full Document)
    cursor.execute(
        "INSERT INTO sops (sop_id, vendor, severity, full_json) VALUES (?, ?, ?, ?)",
        (sop_id, vendor, severity, json.dumps(sop))
    )

    # 2. Prepare for Vector DB & BM25
    bm25_tokenized_corpus.append(simple_tokenize(prose_content))
    batch_docs.append(prose_content)
    batch_ids.append(sop_id)
    batch_metadatas.append({"vendor": vendor, "severity": severity})

conn.commit()

# 3. Insert into ChromaDB
embeddings = embedding_func.encode(batch_docs, normalize_embeddings=True).tolist()
collection.add(
    documents=batch_docs,
    embeddings=embeddings,
    metadatas=batch_metadatas,
    ids=batch_ids
)

print(f"Indexed {collection.count()} SOPs with optimized prose embeddings.")

print("Building BM25 Index...")
bm25 = BM25Okapi(bm25_tokenized_corpus)
del bm25_tokenized_corpus
gc.collect()

In [ ]:
print("Loading SOTA Cross-Encoder: BAAI/bge-reranker-base")
reranker = CrossEncoder('BAAI/bge-reranker-base', device=device)

print("\nLoading Llama-3-8B (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.0
)

print("AI Models Loaded Successfully & Greedy Decoding Enforced.")

In [ ]:
!pip install -q gliner

In [ ]:
from gliner import GLiNER
import numpy as np
import json

# LOAD GLiNER (Zero-Shot Metadata Extractor)
print("Loading GLiNER for localized, zero-shot metadata extraction:")
# Load the model weights
ner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")

ner_model = ner_model.to(device)
def extract_metadata_gliner(query):
    labels = ["telecom vendor", "severity level"]

    # Predict entities directly from the raw string
    entities = ner_model.predict_entities(query, labels)

    filters = {}
    for ent in entities:
        text_val = ent["text"].lower()
        label = ent["label"]

        # Clean and normalize the extracted text to match ChromaDB metadata
        if label == "telecom vendor":
            for v in ["cisco", "juniper", "nokia", "ericsson", "huawei", "palo alto", "fortinet", "ciena"]:
                if v in text_val:
                    filters["vendor"] = v.capitalize()
                    break
        elif label == "severity level":
            for s in ["critical", "major", "minor", "warning"]:
                if s in text_val:
                    filters["severity"] = s.upper()
                    break

    return filters if filters else None

# THE OPTIMIZED QUERY PIPELINE
def process_query(query):
    # Extract Metadata Filters using Local GLiNER
    chroma_filters = extract_metadata_gliner(query)
    print(f"  [Diagnostics] Router applied filters: {chroma_filters}")

    # Dense Retrieval
    query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {query}"], normalize_embeddings=True).tolist()

    dense_res = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        where=chroma_filters
    )

    if not dense_res['ids'][0]:
        return "No matching SOPs found for this specific hardware/fault.", []

    dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}

    # Sparse Retrieval (BM25)
    tokenized_query = simple_tokenize(query)
    sparse_scores = bm25.get_scores(tokenized_query)
    top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

    # Tuned Reciprocal Rank Fusion (RRF) for dense datasets
    fusion_scores = {}
    k = 20 # Aggressively favor the top hits to reduce noise before the Cross-Encoder

    for rank, (doc_id, _) in enumerate(dense_hits.items()):
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    for rank, idx in enumerate(top_sparse_indices):
        doc_id = batch_ids[idx]
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    top_ids = [doc_id for doc_id, _ in sorted_candidates]

    # Fetch FULL JSON from decoupled SQLite storage
    placeholders = ','.join(['?'] * len(top_ids))
    cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
    rows = cursor.fetchall()

    id_to_json = {row[0]: json.loads(row[1]) for row in rows}
    candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]

    # Reranking via Cross-Encoder
    pairs = [[query, doc["search_content"]] for doc in candidate_docs]
    scores = reranker.predict(pairs)
    ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)

    # Expanded Context Window (Taking Top 3 instead of Top 2)
    final_sops = [doc for doc, score in ranked_final][:3]

    # Format Context for Operational Safety
    context_blocks = []
    for sop in final_sops:
        block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
        block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
        for step in sop.get('steps', []):
            block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
        context_blocks.append(block)

    context_string = "\n\n".join(context_blocks)

    # Strict Citation Prompting to prevent Hallucinations
    # Strict XML Chain-of-Thought Prompting
    # Robust Chain-of-Thought Prompting for 4-bit Models
    sys_prompt = (
        "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
        "You must follow these instructions exactly:\n"
        "1. Identify the SINGLE most relevant SOP for the user's issue.\n"
        "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
        "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
        "4. In your final answer, do NOT generate artificial warnings unless the chosen SOP explicitly states them.\n"
        "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command (e.g., `clear ip bgp *` [SOP-123])."
    )

    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Context SOPs:\n{context_string}\n\nUser Issue: {query}"},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # We remove max_length to avoid the Transformers conflict warning
    outputs = llm_pipe(prompt)

    full_out = outputs[0]["generated_text"]
    generated_text = full_out.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

    # Fail proof extraction: Split the string and take everything after the marker
    if "FINAL_ANSWER:" in generated_text:
        answer = generated_text.split("FINAL_ANSWER:")[-1].strip()
    else:
        # Emergency fallback if the model completely ignores formatting
        answer = generated_text

    return answer, final_sops